In [1]:
# Imports

import pandas as pd

In [26]:
# Estaciones meteorologicas

df = pd.read_fwf(
  "./datasets/estaciones_smn.txt",
  encoding="latin-1"
)

df = df.iloc[1:].copy()

df.columns = [
  "nombre",
  "provincia",
  "latitud",
  "longitud",
  "altura",
  "nro",
  "nro_oaci"
]

# Arregla nombres que se cortaron
df = df[~df["nombre"].isin(["ERO", "RO"])].copy()

df["nombre"] = df["nombre"].replace({
  "PRESIDENCIA ROQUE SAENZ PEÑA A": "PRESIDENCIA ROQUE SAENZ PEÑA AERO",
  "ESCUELA DE AVIACION MILITAR AE": "ESCUELA DE AVIACION MILITAR AERO"
})

# Transformar columnas de latitud y longitud

def convertir_coordenada(value):
  partes = str(value).split()

  grados = float(partes[0])
  minutos = float(partes[1])

  signo = -1 if grados < 0 else 1

  return signo * (abs(grados) + minutos / 60)

df["latitud"] = df["latitud"].apply(convertir_coordenada)
df["longitud"] = df["longitud"].apply(convertir_coordenada)

df.to_csv(
  "./datasets/estaciones_meteorologicas.csv",
  index=False
)

df.head()


,nombre,provincia,latitud,longitud,altura,nro,nro_oaci
1,BASE BELGRANO II,ANTARTIDA,-77.866667,-34.616667,256,89034.0,SAYB
2,BASE CARLINI (EX JUBANY),ANTARTIDA,-62.233333,-58.650000,11,89053.0,SAYJ
3,BASE ESPERANZA,ANTARTIDA,-63.383333,-56.983333,24,88963.0,SAYE
4,BASE MARAMBIO,ANTARTIDA,-64.233333,-56.616667,198,89055.0,SAWB
5,BASE ORCADAS,ANTARTIDA,-60.733333,-44.733333,12,88968.0,SAYO


In [2]:
# Registros de temperaturas de los ultimos 365 dias

# Existen casos en donde no se midio ninguna de las dos temperaturas, por ahora se mantienen esas filas igualmente

df = pd.read_fwf(
  "./datasets/registro_temperatura365d_smn.txt",
  encoding="latin-1",
  skiprows=3,
  names=["fecha", "tmax", "tmin", "estacion"],
  dtype={"fecha":str}
)


# Transformar las fechas a formato YYYY-MM-DD

df["fecha"] = pd.to_datetime(
  df["fecha"].str.strip(),
  format="%d%m%Y",
  errors="coerce"
).dt.date

df.to_csv(
  "./datasets/temperaturas_365d.csv",
  index=False
)

df.head()

,fecha,tmax,tmin,estacion
0,2026-09-27,17.5,13.3,AEROPARQUE AERO
1,2026-09-27,18.0,6.3,AZUL AERO
2,2026-09-27,22.8,9.1,BAHIA BLANCA AERO
3,2026-09-27,12.2,3.9,BARILOCHE AERO
4,2026-09-27,-16.1,-20.6,BASE BELGRANO II


In [74]:
# Estadisticas normales 1991 - 2020

df = pd.read_excel(
  "./datasets/estadisticas_normales_1991-2020.xlsx",
  skiprows=4,
  names=["estacion", "valor", "enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre", ],
)

# Limpiar strings
df['estacion'] = df['estacion'].str.replace('*', '')
df['valor'] = df['valor'].str.replace(' (2011-2020)', '')

# Transformar 'S/D'
# Hay bastantes filas sin registros
df = df.replace("S/D", pd.NA)

# df[df['enero'].isnull()].head(20)

df.to_csv(
  "./datasets/estadisticas_normales_1991-2020.csv",
  index=False
)

df.head()

,estacion,valor,enero,febrero,marzo,abril,mayo,junio,julio,agosto,septiembre,octubre,noviembre,diciembre
0,LA QUIACA OBSERVATORIO,Temperatura (°C),13.2,13,12.8,11.3,7.3,4.8,4.5,7,10,12.4,13.4,13.9
1,LA QUIACA OBSERVATORIO,Temperatura máxima (°C),20.6,20.4,20.6,20.3,17.8,16.3,16.1,18,20,21.7,22.5,22.2
2,LA QUIACA OBSERVATORIO,Temperatura mínima (°C),7.7,7.6,6.6,3.1,-2.5,-5.7,-6.2,-4,-0.4,3.3,5.5,7.3
3,LA QUIACA OBSERVATORIO,Humedad relativa (%),62.6,63.2,60.3,46,32.6,27.4,25.7,26.7,32.1,42.4,48.6,55.8
4,LA QUIACA OBSERVATORIO,Velocidad del Viento (km/h),6.5,6.8,6.7,5.5,4.8,5.5,5.9,6.7,7.9,7.9,7.7,7.1
